# Cyber Trend Forecasting - B-MTGNN Pipeline

**Vignette Description:**
This notebook acts as the Pipeline Dashboard for the Bayesian Multivariate Time Graph Neural Network (B-MTGNN). It handles the end-to-end workflow
1.  **Setup:** Automatically detects if running on Google Colab or the NATO Local Server.
2.  **Data Preprocessing** Executes the data preparation scripts to construct the Adjacency Matrix and format the time series
3.  **Training:** Executes `Train.py` to train the Bayesian-MTGNN.
4.  **Evaluation:** Executes `Evaluation_Metrics.py` to evaluate.

## Universal Setup

This cell handles the setup logic: mounting Google Drive if on Colab, or using relative paths if on the GPU server

In [1]:
import os
import sys
import contextlib

# --- CONFIGURATION ---
# If on Colab, ensure you have set the Secret 'project_path' to your Project Root
# e.g., /content/drive/MyDrive/NATO_Project_Root

try:
    # 1. CHECK ENVIRONMENT: Try to import Colab-specific modules
    from google.colab import drive, userdata
    IS_COLAB = True
    print("--- Detected Environment: GOOGLE COLAB ---")
except ImportError:
    IS_COLAB = False
    print("--- Detected Environment: LOCAL SERVER (NATO/BBK) ---")

# 2. PATH SETUP
if IS_COLAB:
    # --- COLAB SETUP ---
    try:
        drive.mount('/content/drive')

        # Get the PARENT ROOT from Secrets
        PROJECT_ROOT = userdata.get('project_path')
        if not PROJECT_ROOT:
            raise ValueError("Secret 'project_path' is missing.")

    except Exception as e:
        print(f"COLAB SETUP ERROR: {e}")
        # Stop execution if we can't find the files
        raise e
else:
    # --- LOCAL SETUP ---
    # We assume this notebook is in 'Notebooks/', so Root is one level up
    # os.getcwd() returns /.../Project_Root/Notebooks
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# 3. CONFIGURE WORKING DIRECTORY
# We need to run scripts from inside 'Transformer_Pipeline'
PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'Data_Pipeline')

if not os.path.exists(PIPELINE_DIR):
    raise FileNotFoundError(f"Could not find pipeline folder at: {PIPELINE_DIR}")

# Change CWD to the pipeline folder so scripts can find config/data
os.chdir(PIPELINE_DIR)
print(f"SUCCESS: Working Directory set to: {os.getcwd()}")

# 4. ADD PROJECT ROOT TO SYS.PATH
# This allows us to import 'Notebooks.colab_utils' if needed
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added Project Root to sys.path: {PROJECT_ROOT}")

# 5. INSTALL DEPENDENCIES (COLAB ONLY)
if IS_COLAB:
    print("\n--- Installing Dependencies (Colab Only) ---")
    # We import the helper script from the sibling folder
    try:
        from Notebooks import Colab_Utils
        # This creates a temporary black hole for the print statements
        with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
            Colab_Utils.setup_environment(PIPELINE_DIR)
        print("Dependencies verified successfully!")
    except ImportError:
        print("WARNING: Could not import Notebooks.colab_utils. Skipping auto-install.")

--- Detected Environment: LOCAL SERVER (NATO/BBK) ---
SUCCESS: Working Directory set to: /mnt/c/Workspace/Research/Cyber-trend-forecasting/Data_Pipeline
Added Project Root to sys.path: /mnt/c/Workspace/Research/Cyber-trend-forecasting


### Imports

In [2]:
import os
import glob
from IPython.display import display, Image
from pathlib import Path


### Graph & Data Preprocessing

This step ensures the raw CSV data is properly formatted for the Graph Neural Network. It processes the historical time series and constructs the underlying spatial graph (Adjacency Matrix) linking threats to their respective solutions.

In [3]:
# Ensure working directory is the project root for data pipeline execution
os.chdir(PROJECT_ROOT)

print("--- Processing Historical Datasets ---")

# Using !python instead of %run ensures argparse receives the strings correctly
!python Data_Pipeline/Prep_Unsmoothed_Data.py --input_file Data_Preparation/Cyber_Trend_Forecasting_All_v2.csv
!python Data_Pipeline/Prep_Unsmoothed_Data.py --input_file Data_Preparation/Cyber_Trend_Forecasting_All_v2_1.csv
!python Data_Pipeline/Prep_Unsmoothed_Data.py --input_file Data_Preparation/Cyber_Trend_Forecasting_All_v2_2_sarimax.csv

--- Processing Historical Datasets ---
--- Initiating Phase 1: B-MTGNN Data Preparation ---
Loading raw dataset from: Data_Preparation/Cyber_Trend_Forecasting_All_v2.csv
Original dataset shape: (174, 1232) (Months x Columns)
Filtered dataset shape: (174, 123) (Months x Columns)
Applying np.clip to sanitize data (setting minimum bound to 0.0)...
Comparing new data against existing working file...
Exporting legacy B-MTGNN matrix to: Processed_Data/B-MTGNN/sm_data.txt
Exporting VisionTS++ CSV to: Processed_Data/VisionTS/Mark3_Clipped_Data.csv
Archiving B-MTGNN matrix to: Processed_Data/B-MTGNN/Archive/sm_data_Cyber_Trend_Forecasting_All_v2.txt
Archiving VisionTS++ CSV to: Processed_Data/VisionTS/Archive/clipped_Cyber_Trend_Forecasting_All_v2.csv

SUCCESS: Phase 1 Data preparation and archiving complete.
--- Initiating Phase 1: B-MTGNN Data Preparation ---
Loading raw dataset from: Data_Preparation/Cyber_Trend_Forecasting_All_v2_1.csv
Original dataset shape: (174, 1232) (Months x Columns)


### Hyperparameter Grid Search && Train B-MTGNN Model
We execute a random grid search to identify the optimal spatial and temporal graph configurations for the new Mark 3 dataset topology.

- Input: The processed `.txt` data and `graph.csv` adjacency mappings.
- Output: `hp.txt` (Saved optimal hyperparameters).

Note: We use `%run` to execute this natively within the notebook, providing live iteration updates.

We now execute the training script.

- Input: The processed `.txt` data and `graph.csv` adjacency mappings.

- Output: `o_model.pt` (Saved model weights).

Note: You can easily adjust the `--epochs`, `--batch_size`, and `--num_nodes` directly in the `%run` command below for demonstration purposes.

We use `%run` to execute the script natively within the notebook namespace,
providing live epoch updates and progress outputs.

#### Run 1: Original Mark 3 (YouTube + Reddit)

In [5]:
%%capture run_1_output
# Cast the string to a Path object to enable the / concatenation operator
PROJECT_ROOT_PATH = Path(PROJECT_ROOT)

# Change to B-MTGNN directory for training
os.chdir(PROJECT_ROOT_PATH / 'B-MTGNN')

# Construct absolute paths to prevent relative directory errors
data_file_v2 = str(PROJECT_ROOT_PATH / "Processed_Data" / "B-MTGNN" / "Archive" / "sm_data_Cyber_Trend_Forecasting_All_v2.txt")
model_v2 = "model/Bayesian/model_v2.pt"

print("--- Run 1: Original Mark 3 (YouTube + Reddit) ---")
%run train_test.py --data $data_file_v2 --epochs 50 --batch_size 16 --num_nodes 123
%run train.py --data $data_file_v2 --save $model_v2 --epochs 500 --batch_size 16 --num_nodes 123

#### Run 2: Mark 3 (Reddit Only)

In [6]:
%%capture run_2_output

data_file_v2_1 = str(PROJECT_ROOT_PATH / "Processed_Data" / "B-MTGNN" / "Archive" / "sm_data_Cyber_Trend_Forecasting_All_v2_1.txt")
model_v2_1 = "model/Bayesian/model_v2_1.pt"

print("--- Run 2: Mark 3 (Reddit Only) ---")
%run train_test.py --data $data_file_v2_1 --epochs 50 --batch_size 16 --num_nodes 123
%run train.py --data $data_file_v2_1 --save $model_v2_1 --epochs 500 --batch_size 16 --num_nodes 123

### Run 3: Mark 3 SARIMAX (Reddit + Hackmageddon | Legacy Model)")

In [7]:
%%capture run_3_output

data_file_v2_2 = str(PROJECT_ROOT_PATH / "Processed_Data" / "B-MTGNN" / "Archive" / "sm_data_Cyber_Trend_Forecasting_All_v2_2_sarimax.txt")
model_v2_2_legacy = "model/Bayesian/model_v2_2_legacy.pt"

print("--- Run 3: Mark 3.2 SARIMAX (Legacy Overfit Run) ---")
%run train_test.py --data $data_file_v2_2 --epochs 50 --batch_size 16 --num_nodes 123
%run Legacy/train.py --data $data_file_v2_2 --save $model_v2_2_legacy --epochs 500 --batch_size 16 --num_nodes 123

#### Run 4: Mark 3 SARIMAX (Reddit + Hackmageddon | Updated Model )

In [8]:
%%capture run_4_output

model_v2_2_updated = "model/Bayesian/model_v2_2_updated.pt"

print("--- Run 4: Mark 3.3 SARIMAX (Updated Patched Run) ---")
%run train.py --data $data_file_v2_2 --save $model_v2_2_updated --epochs 500 --batch_size 16 --num_nodes 123

### Multi-Horizon Evaluation Extraction

Execution: Loads the trained B-MTGNN model and historical testing dataset to compute standard forecasting error metrics (RSE, RAE, MAE, CORR) across discrete temporal horizons (3, 6, 12, and 24 months).

Outputs: Generates a structured evaluation table and saves it as `graph_evaluation_results.csv` for comparative analysis.

In [11]:
import os
from pathlib import Path

# Ensure PROJECT_ROOT_PATH is available in memory
if 'PROJECT_ROOT_PATH' not in locals():
    PROJECT_ROOT_PATH = Path(PROJECT_ROOT)

# KEEP working directory in B-MTGNN so Zaid's util.py can find 'data/graph.csv'
os.chdir(PROJECT_ROOT_PATH / 'B-MTGNN')

print("============================================================")
print("--- Baseline: Mark 2 (Feb 2026) ---")
print("============================================================")

# FOOLPROOF FIX: Combine both required paths into a single environment variable
env_path = f"{PROJECT_ROOT_PATH}:{PROJECT_ROOT_PATH / 'B-MTGNN'}"

# Use absolute paths constructed by pathlib to prevent ANY relative path failures
eval_script = PROJECT_ROOT_PATH / "Data_Pipeline" / "Evaluation_Metrics.py"
model_path = PROJECT_ROOT_PATH / "Processed_Data" / "B-MTGNN" / "Archive_Mark2" / "o_model.pt"
data_path = PROJECT_ROOT_PATH / "Processed_Data" / "B-MTGNN" / "Archive_Mark2" / "sm_data.txt.txt"

# Execute the shell command using the absolute paths
!CUDA_VISIBLE_DEVICES="" PYTHONPATH="{env_path}" python "{eval_script}" --experiment_tag "_baseline_mark2" --model_file "{model_path}" --data_file "{data_path}"

print("\n============================================================")
print("--- Generating Evaluation Metrics for Current Runs ---")
print("============================================================")

# Evaluate Run 1 (Original Mark 3: YouTube + Reddit)
%run ../Data_Pipeline/Evaluation_Metrics.py --experiment_tag "_v2" --model_file "model/Bayesian/model_v2.pt" --data_file "../Processed_Data/B-MTGNN/Archive/sm_data_Cyber_Trend_Forecasting_All_v2.txt"

# Evaluate Run 2 (Mark 3: Reddit Only)
%run ../Data_Pipeline/Evaluation_Metrics.py --experiment_tag "_v2_1" --model_file "model/Bayesian/model_v2_1.pt" --data_file "../Processed_Data/B-MTGNN/Archive/sm_data_Cyber_Trend_Forecasting_All_v2_1.txt"

# Evaluate Run 3 (SARIMAX on Legacy Architecture)
%run ../Data_Pipeline/Evaluation_Metrics.py --experiment_tag "_v2_2_legacy" --model_file "model/Bayesian/model_v2_2_legacy.pt" --data_file "../Processed_Data/B-MTGNN/Archive/sm_data_Cyber_Trend_Forecasting_All_v2_2_sarimax.txt"

# Evaluate Run 4 (SARIMAX on Updated Architecture)
%run ../Data_Pipeline/Evaluation_Metrics.py --experiment_tag "_v2_2_updated" --model_file "model/Bayesian/model_v2_2_updated.pt" --data_file "../Processed_Data/B-MTGNN/Archive/sm_data_Cyber_Trend_Forecasting_All_v2_2_sarimax.txt"

--- Baseline: Mark 2 (Feb 2026) ---
Graph loaded with 26 attacks...
123 columns loaded...
Adjacency created...
Evaluation Results:
  Horizon      RSE      RAE       MAE      CORR
 3 Months 1.106695 1.183602 13.162534 -0.092533
 6 Months 1.203059 1.372252 15.586897  0.058079
12 Months 1.396637 1.487748 16.806190  0.001484
24 Months 1.721855 1.506960 17.162371 -0.017601
  Overall 1.320135 1.286154 16.384945  0.017938

Saved results table to Data_Pipeline/Results/graph_evaluation_results_baseline_mark2.csv

--- Generating Evaluation Metrics for Current Runs ---
Graph loaded with 26 attacks...
123 columns loaded...
Adjacency created...
Evaluation Results:
  Horizon      RSE      RAE      MAE      CORR
 3 Months 1.177692 1.130230 4.953329 -0.134908
 6 Months 1.094613 1.102028 4.410926 -0.077881
12 Months 1.176507 1.247642 4.434272  0.119683
24 Months 0.982495 1.065990 6.399807  0.114766
  Overall 1.065851 1.101976 7.225456 -0.014120

Saved results table to Data_Pipeline/Results/graph_evalua